Simulating Light Curves

In [ ]:
import numpy as np
import pandas as pd
import os
import glob
from eztao.carma import DRW_term
from eztao.ts import gpSimByTime

# --- 1. Simulation Logic ---
def simulate_realistic_sdss_lightcurve(time_array, sdss_errors, tau, sigma, mean_mag):
    # Initialize the DRW kernel
    drw_kernel = DRW_term(np.log(sigma), np.log(tau))
    
    # Generate the underlying "True" signal
    t_out, y_true, model_err = gpSimByTime(drw_kernel, 10000, time_array, nLC=1, log_flux=False)
    
    # Shift to the correct mean magnitude
    y_true = y_true + mean_mag
    
    # Inject FRESH random noise based on the SDSS errors
    injected_noise = np.random.normal(loc=0.0, scale=sdss_errors)
    y_mock_observed = y_true + injected_noise
    
    return y_mock_observed, model_err

# --- 2. Setup Paths ---
lc_folder = r'data/cleaned_light_curves'
output_folder = r'data/simulated_DRW' # <--- Validation Folder
master_data_path = r'data/DRW_data.pkl'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# --- 3. Load Master Parameters ---
master_df = pd.read_pickle(master_data_path)

# --- CRITICAL: Set the Seed ---
# This guarantees that 'y_true' will be a DIFFERENT path than the training set
# even though tau and sigma are the same.
np.random.seed(888) 

print(f"Starting FULL validation simulation for {len(master_df)} quasars...")

# --- 4. Main Loop ---
success_count = 0

for index, row in master_df.iterrows():
    # Get ID and physical parameters
    qso_id = str(int(row['dbID']))
    tau = row['tau']
    sigma = row['sigma']
    
    # Skip if parameters are invalid
    if pd.isna(tau) or pd.isna(sigma):
        continue

    # Find the corresponding original file to get timestamps/errors
    matching_files = glob.glob(os.path.join(lc_folder, f"{qso_id}*"))
    
    if matching_files:
        file_path = matching_files[0]
        try:
            # Read real data structure
            df_real = pd.read_csv(file_path, sep=None, engine='python')
            
            t_real = df_real['MJD_r'].values
            sdss_err = df_real['magr_err'].values
            mean_mag = df_real['mag_r'].mean()
            
            # Run simulation
            mock_mag, model_err = simulate_realistic_sdss_lightcurve(
                t_real, sdss_err, tau, sigma, mean_mag
            )
            
            # Save Validation File
            sim_df = pd.DataFrame({
                'MJD': t_real,
                'mag_r_sim': mock_mag, # The NEW stochastic path
                'magr_err': sdss_err,
                'model_err': model_err
            })
            
            sim_df.to_csv(os.path.join(output_folder, f"{qso_id}.csv"), index=False)
            success_count += 1
            
            if success_count % 500 == 0:
                print(f"Processed {success_count}/{len(master_df)} files...")
                
        except Exception as e:
            print(f"Error on ID {qso_id}: {e}")

print(f"Validation simulation complete! {success_count} files saved in: {output_folder}")